<a href="https://colab.research.google.com/github/SethuMeenal/first-repo/blob/main/Subjective%20Assignment%3A%20Intermediate%20SQL%20Querying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

TASK-1

In [3]:
query = """SELECT customerid,count(orderID) as order_count,sum(freight) as total_freight,
avg(freight) as avg_freight from orders group by customerid
order by sum(freight) desc"""
sql_result1 = pd.read_sql(query,conn)
print(sql_result1.head(10))

  customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


TASK-2

In [6]:
query = """Select customerid,COUNT(orderid) as high_freight_orders from orders
WHERE FREIGHT > 50 group by customerid"""
sql_resultA = pd.read_sql_query(query,conn)
print("count of high freight orders of customer")
print(sql_resultA)
query = """Select customerid,SUM(freight) as total_freight from orders
group by customerid HAVING SUM(freight) > 500"""
sql_resultB = pd.read_sql_query(query,conn)
print("customers whose total freight exceeding 500")
print(sql_resultB)

count of high freight orders of customer
   customerID  high_freight_orders
0       ALFKI                    2
1       ANTON                    2
2       AROUT                    2
3       BERGS                   11
4       BLAUS                    1
..        ...                  ...
69      WANDK                    2
70      WARTH                    6
71      WELLI                    1
72      WHITC                    7
73      WOLZA                    1

[74 rows x 2 columns]
customers whose total freight exceeding 500
   customerID  total_freight
0       BERGS        1559.52
1       BLONP         623.66
2       BONAP        1357.87
3       BOTTM         793.95
4       EASTC         832.34
5       ERNSH        6205.39
6       FOLIG         637.94
7       FOLKO        1678.08
8       FRANK        1403.44
9       GODOS         568.27
10      GREAL        1087.61
11      HANAR         724.77
12      HILAA        1259.16
13      HUNGO        2755.24
14      KOENE         813.68
15      

QueryA is doing filter before aggregation(where clause) to choose ony orders whose any one order frieght is greater than 50, whereas QueryB is doing filter after aggregation(having clause) to choose only customers whose total freight  of all their orders exceed 500.

TASK-3

In [15]:
query = """Select c.companyName,c.country,count(o.orderID) as order_count,sum(freight) as total_freight
from orders o JOIN customers c ON o.customerID = c.customerID
WHERE FREIGHT > 50 group by c.customerid"""
sql_resultC = pd.read_sql_query(query,conn)
print("details of customer who has atleast placed 1 order")
print(sql_resultC)
query = """Select c.companyName,c.country,count(o.orderID) as order_count,nvl(sum(freight) as total_freight
from customers c LEFT JOIN orders o ON o.customerID = c.customerID
group by c.customerid order by sum(freight)"""
sql_resultD = pd.read_sql_query(query,conn)
print("details of all customers including ones who never placed any order")
print(sql_resultD)

details of customer who has atleast placed 1 order
                companyName  country  order_count  total_freight
0       Alfreds Futterkiste  Germany            2         130.55
1   Antonio Moreno Taquería   Mexico            2         143.27
2           Around the Horn       UK            2         219.29
3        Berglunds snabbköp   Sweden           11        1471.84
4   Blauer See Delikatessen  Germany            1          53.83
..                      ...      ...          ...            ...
69        Die Wandernde Kuh  Germany            2         177.29
70           Wartian Herkku  Finland            6         677.53
71   Wellington Importadora   Brazil            1          55.23
72     White Clover Markets      USA            7        1212.22
73           Wolski  Zajazd   Poland            1          80.65

[74 rows x 4 columns]
details of all customers including ones who never placed any order
                             companyName  country  order_count  total_freight
0

Reason: QueryC is doing inner join because only common rows in both tables is what we need (for bringing customer who placed atleast one order) whereas QueryD is doing left join because all customers in left(customer) table is needed(irrespective of whether they've placed any orders or not)